Judge saved model responses and record case-level scores, criterion decisions and token usage.


In [ ]:
from pathlib import Path
import os
_repository_root = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'data').is_dir() and (p / 'experiments/clinical_protocols').is_dir()))
os.chdir(_repository_root / 'evaluations/geval_judges/geval_generated_reasoning')
from pathlib import Path
import os
_REPOSITORY_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()))
if os.name == 'nt' and (not str(_REPOSITORY_ROOT).startswith('\\\\?\\')):
    _REPOSITORY_ROOT = Path('\\\\?\\' + str(_REPOSITORY_ROOT))
_NOTEBOOK_DIR = _REPOSITORY_ROOT / 'evaluations/geval_judges/geval_generated_reasoning'
os.chdir(_NOTEBOOK_DIR)


In [ ]:
import hashlib
import json
import math
import os
import re
import time
from pathlib import Path
from typing import Any, Dict, List
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
try:
    from openai import OpenAI
except ImportError:
    OpenAI = None
pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.max_colwidth', 90)


In [ ]:
ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'data').is_dir() and (p / 'experiments/clinical_protocols').is_dir()))
DESIGN = 'noroles'
ARMS = {'delphi': ROOT / 'evaluations/geval_judges/source_protocols/delphi/llm_as_judge/delphi_base_noroles_judge.csv', 'baseline_gpt': ROOT / 'evaluations/geval_judges/source_protocols/baseline_gpt/llm_as_judge/baseline_noroles_gpt_judge.csv', 'baseline_qwen': ROOT / 'evaluations/geval_judges/source_protocols/baseline_qwen/llm_as_judge/baseline_noroles_qwen_judge.csv', 'baseline_deepseekv4': ROOT / 'evaluations/geval_judges/source_protocols/baseline_deepseekv4/llm_as_judge/baseline_noroles_deepseekv4_judge.csv'}
JOIN_KEY = 'Vignette'
MODEL = 'gpt-4.1-mini-2025-04-14'
API_KEY = ''
SCALE_LO, SCALE_HI = (1, 5)
TEMPERATURE_SCORE = 0.0
MAX_TOKENS_SCORE = 5
TOP_LOGPROBS = 20
TEMPERATURE_STEPS = 0.3
MAX_TOKENS_STEPS = 900
REQUEST_TIMEOUT = 60
BASE_SLEEP_SECONDS = 2
MAX_RETRIES = 6
RUN = True
SAMPLE_N = None
STEPS_JSON = Path('geval_steps.json')
OUT_CSV = Path('geval_scores.csv')


In [ ]:
def read_csv(path):
    d = pd.read_csv(path, sep=';', low_memory=False)
    d.columns = [c.lstrip('\ufeff') for c in d.columns]
    return d

def strip_author(text):
    if not text:
        return ''
    return re.split('(?im)^\\s*Protocol Author\\s*:', str(text), maxsplit=1)[0].rstrip()

def n_words(text):
    return len(re.findall('\\w+', str(text)))
frames = []
for arm, path in ARMS.items():
    d = read_csv(path)
    missing = [c for c in (JOIN_KEY, 'case_text', 'winner_protocol') if c not in d.columns]
    if missing:
        raise ValueError(f'{arm}: нет колонок {missing}')
    d = d[[JOIN_KEY, 'case_text', 'winner_protocol']].copy()
    d = d[d['winner_protocol'].notna()]
    d['arm'] = arm
    d['protocol'] = d['winner_protocol'].map(strip_author)
    d['words'] = d['protocol'].map(n_words)
    frames.append(d[[JOIN_KEY, 'arm', 'case_text', 'protocol', 'words']])
protocols = pd.concat(frames, ignore_index=True)
if SAMPLE_N:
    keep = protocols[JOIN_KEY].drop_duplicates().head(SAMPLE_N)
    protocols = protocols[protocols[JOIN_KEY].isin(keep)].reset_index(drop=True)
print(f'рук: {protocols.arm.nunique()}   кейсов: {protocols[JOIN_KEY].nunique()}   протоколов: {len(protocols)}')
protocols.groupby('arm').agg(n=('protocol', 'size'), mean_words=('words', 'mean')).round(0)


In [ ]:
judge_client = None
llm_cache: Dict[str, Any] = {}

def _cache_key(*parts):
    raw = json.dumps({'model': MODEL, 'parts': list(parts)}, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(raw.encode('utf-8')).hexdigest()

def _client():
    global judge_client
    if judge_client is None:
        if not API_KEY:
            raise RuntimeError('OPENAI_API_KEY не задан в окружении')
        judge_client = OpenAI(api_key=API_KEY)
    return judge_client

def call_text(system_text, user_text, temperature, max_tokens):
    key = _cache_key('text', system_text, user_text, temperature, max_tokens)
    if key in llm_cache:
        return llm_cache[key]
    last = ''
    for attempt in range(MAX_RETRIES):
        try:
            r = _client().chat.completions.create(model=MODEL, messages=[{'role': 'system', 'content': system_text}, {'role': 'user', 'content': user_text}], temperature=temperature, max_tokens=max_tokens, timeout=REQUEST_TIMEOUT)
            out = r.choices[0].message.content.strip()
            llm_cache[key] = out
            return out
        except Exception as e:
            last = str(e)
            time.sleep(BASE_SLEEP_SECONDS * (attempt + 1))
    raise RuntimeError(f'call_text failed after {MAX_RETRIES} retries: {last}')

def call_score(prompt_text):
    key = _cache_key('score', prompt_text)
    if key in llm_cache:
        return llm_cache[key]
    last = ''
    for attempt in range(MAX_RETRIES):
        try:
            r = _client().chat.completions.create(model=MODEL, messages=[{'role': 'system', 'content': prompt_text}], temperature=TEMPERATURE_SCORE, max_tokens=MAX_TOKENS_SCORE, logprobs=True, top_logprobs=TOP_LOGPROBS, timeout=REQUEST_TIMEOUT)
            dist = _dist_from_logprobs(r)
            out = {'dist': dist, 'text': (r.choices[0].message.content or '').strip()}
            llm_cache[key] = out
            return out
        except Exception as e:
            last = str(e)
            time.sleep(BASE_SLEEP_SECONDS * (attempt + 1))
    raise RuntimeError(f'call_score failed after {MAX_RETRIES} retries: {last}')

def _dist_from_logprobs(response):
    content = getattr(response.choices[0], 'logprobs', None)
    if content is None or not getattr(content, 'content', None):
        return {}
    for token in content.content:
        dist = {}
        for alt in token.top_logprobs:
            t = alt.token.strip()
            if t.isdigit() and SCALE_LO <= int(t) <= SCALE_HI:
                dist[int(t)] = dist.get(int(t), 0.0) + math.exp(alt.logprob)
        if dist:
            z = sum(dist.values())
            return {s: p / z for s, p in dist.items()}
    return {}

def weighted_score(dist):
    return sum((s * p for s, p in dist.items())) if dist else np.nan

def argmax_score(dist):
    return max(dist, key=dist.get) if dist else np.nan

def dist_entropy(dist):
    return -sum((p * math.log(p) for p in dist.values() if p > 0)) if dist else np.nan


In [ ]:
N_METRICS_MIN, N_METRICS_MAX = (4, 8)
TITLE = 'G-Eval (полный Auto-CoT): Delphi vs matched baselines, noroles'
PNG_NAME = 'geval_generated_reasoning.png'


In [ ]:
_DESIGN_SYSTEM = 'You are an expert in the design of evaluation protocols for\nnatural language generation. You return JSON only.'
_DESIGN_PROMPT = f'Task description:\n\nA model is given a clinical case consisting of a vignette, review of systems, past medical\nhistory, physical examination and test results. The model must write a single structured\nclinical protocol in plain text with exactly these fields, in this order: Age, Sex, Complaint,\nAnalysis, Required diagnostic equipment/tests, Conclusion, Emergency type, Subclinical\nspecialization, Recommendations, Notes.\n\nYour task is to design the evaluation metrics a human rater should use to judge the quality of\nsuch a protocol, and, for each metric, the concrete steps the rater must follow to apply it.\n\nRequirements:\n- Propose between {N_METRICS_MIN} and {N_METRICS_MAX} metrics.\n- The metrics must be distinct and jointly cover what matters for this task.\n- Each metric is rated on an integer scale from {SCALE_LO} to {SCALE_HI}.\n- Each definition is one paragraph and states explicitly what the metric does NOT cover.\n- Each metric has 3 to 5 evaluation steps; each step is one checkable sentence.\n\nReturn JSON only:\n{{\n  "metrics": [\n    {{"name": "...", "definition": "...", "steps": ["...", "..."]}}\n  ]\n}}'

def _parse_metrics(text):
    m = re.search('\\{[\\s\\S]*\\}', str(text))
    data = json.loads(m.group(0)) if m else {}
    out = []
    for item in data.get('metrics', []):
        steps = item.get('steps') or []
        if isinstance(steps, str):
            steps = [steps]
        out.append({'name': str(item.get('name', '')).strip(), 'definition': str(item.get('definition', '')).strip(), 'steps': '\n'.join((f'{i}. {s}' for i, s in enumerate(steps, 1)))})
    return [m for m in out if m['name'] and m['definition'] and m['steps']]
if STEPS_JSON.exists():
    METRICS = json.loads(STEPS_JSON.read_text(encoding='utf-8'))
    print(f'метрики загружены из {STEPS_JSON} ({len(METRICS)}) — генерация пропущена')
else:
    raw = call_text(_DESIGN_SYSTEM, _DESIGN_PROMPT, TEMPERATURE_STEPS, 3000)
    METRICS = _parse_metrics(raw)
    if not N_METRICS_MIN <= len(METRICS) <= N_METRICS_MAX:
        raise ValueError(f'модель вернула {len(METRICS)} метрик, ожидалось {N_METRICS_MIN}-{N_METRICS_MAX}; сырой ответ:\n{raw[:1500]}')
    STEPS_JSON.write_text(json.dumps(METRICS, ensure_ascii=False, indent=1), encoding='utf-8')
    print(f'метрики сгенерированы и заморожены в {STEPS_JSON}')
for m in METRICS:
    print('\n===', m['name'], '===')
    print(m['definition'][:400])
    print(m['steps'])


In [ ]:
def build_geval_prompt(metric, case_text, protocol):
    return f"You will be given one clinical protocol written in response to a clinical case.\n\nYour task is to rate the protocol on one metric.\n\nPlease make sure you read and understand these instructions carefully. Please keep this\ndocument open while reviewing, and refer to it as needed.\n\nEvaluation Criteria:\n\n{metric['name']} ({SCALE_LO}-{SCALE_HI}) - {metric['definition']}\n\nEvaluation Steps:\n\n{metric['steps']}\n\nClinical case:\n<<<CASE>>>\n{case_text}\n<<<END CASE>>>\n\nProtocol:\n<<<PROTOCOL>>>\n{protocol}\n<<<END PROTOCOL>>>\n\nEvaluation Form (scores ONLY):\n\n- {metric['name']}:"


In [ ]:
if RUN:
    records = []
    todo = [(row, m) for _, row in protocols.iterrows() for m in METRICS]
    for row, metric in tqdm(todo, desc='G-Eval'):
        prompt = build_geval_prompt(metric, row['case_text'], row['protocol'])
        out = call_score(prompt)
        dist = out['dist']
        records.append({JOIN_KEY: row[JOIN_KEY], 'arm': row['arm'], 'metric': metric['name'], 'score': weighted_score(dist), 'argmax': argmax_score(dist), 'entropy': dist_entropy(dist), 'n_support': len(dist), 'words': row['words'], 'raw': out['text'], 'dist_json': json.dumps({str(k): round(v, 6) for k, v in sorted(dist.items())})})
    scores = pd.DataFrame(records)
    scores.to_csv(OUT_CSV, sep=';', index=False, encoding='utf-8-sig')
    print(f'сохранено: {OUT_CSV}  ({len(scores)} строк = {protocols.shape[0]} протоколов x {len(METRICS)} метрик)')
else:
    print('RUN=False — прогон пропущен. Выставьте RUN=True и OPENAI_API_KEY в окружении.')


In [ ]:
scores = pd.read_csv(OUT_CSV, sep=';')
pivot = scores.pivot_table(index='arm', columns='metric', values='score', aggfunc='mean')
pivot['MEAN'] = pivot.mean(axis=1)
print('Средний G-Eval балл по рукам и метрикам (шкала %d-%d):' % (SCALE_LO, SCALE_HI))
display(pivot.round(3))
print('\nРазрешающая способность: вероятностный балл против «сырого» целого')
res = scores.groupby('metric').agg(mean_weighted=('score', 'mean'), mean_argmax=('argmax', 'mean'), sd_weighted=('score', 'std'), sd_argmax=('argmax', 'std'), mean_entropy=('entropy', 'mean'), frac_saturated=('score', lambda s: float((s >= SCALE_HI - 0.01).mean())))
res['sd_gain'] = res['sd_weighted'] / res['sd_argmax'].replace(0, np.nan)
display(res.round(4))
print('sd_gain >> 1 -> вероятностное взвешивание вернуло дисперсию, которой нет у целых оценок;')
print('frac_saturated -> доля протоколов, упёршихся в потолок шкалы (эффект насыщения).')


In [ ]:
from scipy import stats
rows = []
for base in [a for a in scores.arm.unique() if a != 'delphi']:
    for m in scores.metric.unique():
        d = scores[(scores.arm == 'delphi') & (scores.metric == m)][[JOIN_KEY, 'score', 'words']]
        b = scores[(scores.arm == base) & (scores.metric == m)][[JOIN_KEY, 'score', 'words']]
        j = d.merge(b, on=JOIN_KEY, suffixes=('_d', '_b'))
        if not len(j):
            continue
        diff = j.score_d - j.score_b
        t = stats.wilcoxon(j.score_d, j.score_b) if diff.abs().sum() > 0 else None
        r = stats.pearsonr(diff, j.words_d - j.words_b) if diff.std() > 0 else (np.nan, np.nan)
        rows.append({'baseline': base, 'metric': m, 'n': len(j), 'delphi': j.score_d.mean(), 'base': j.score_b.mean(), 'delta': diff.mean(), 'p_wilcoxon': t.pvalue if t else np.nan, 'r(delta, dwords)': r[0]})
delta = pd.DataFrame(rows)
print('Delphi - baseline по метрикам. r(delta, dwords) — связь преимущества с разницей в длине:')
display(delta.round(4))


In [ ]:
import matplotlib.pyplot as plt
order = list(pivot.columns.drop('MEAN'))
arms = list(pivot.index)
x = np.arange(len(order))
w = 0.8 / len(arms)
fig, ax = plt.subplots(figsize=(max(8, 1.4 * len(order)), 4.5))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')
for i, arm in enumerate(arms):
    ax.bar(x + i * w - 0.4 + w / 2, pivot.loc[arm, order].values, width=w, label=arm)
ax.set_xticks(x)
ax.set_xticklabels(order, rotation=20, ha='right', fontsize=9)
ax.set_ylim(SCALE_LO, SCALE_HI)
ax.set_ylabel(f'G-Eval score ({SCALE_LO}-{SCALE_HI})', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(frameon=False, fontsize=9, ncol=len(arms))
ax.set_title(TITLE, fontsize=12, fontweight='bold', pad=10)
fig.tight_layout()
fig.savefig(PNG_NAME, dpi=170, facecolor='white')
plt.show()
